In [1]:
# ============================================================================
# CELL 1: Setup and Data Loading
# ============================================================================
import numpy as np
import pandas as pd
import os
import torch
import torchvision
import torch.nn as nn
import pytorch_lightning as pl
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import v2
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pytorch_lightning import Trainer
from tqdm import tqdm
import cv2
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# Load data
train_file_path = "/kaggle/input/csiro-biomass/train.csv"
test_file_path = "/kaggle/input/csiro-biomass/test.csv"

train_pd = pd.read_csv(train_file_path)
test_pd_original = pd.read_csv(test_file_path)  # Keep original for submission
test_pd = test_pd_original.copy()  # Working copy for predictions

print(f"Train shape: {train_pd.shape}")
print(f"Test shape: {test_pd.shape}")

Train shape: (1785, 9)
Test shape: (5, 3)


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/resnet50/resnet50_weights_tf_dim_ordering_tf_kernels.h5
/kaggle/input/resnet50/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5
/kaggle/input/resnet50/imagenet_class_index.json
/kaggle/input/vit-huge-plus-patch16-dinov3-lvd1689m/pytorch/default/1/vit_huge_plus_patch16_dinov3.lvd1689m_backbone.pth
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/full_image_88545560_22x8_v12_epoch_00153.labels.20251208.txt
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/README.md
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/info.json
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/taxonomy_release.20251208.txt
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/full_image_88545560_22x8_v12_epoch_00153.pt
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/geofence_release.20251208.json
/kaggle/input/baseline-dinov3/pytorch/default/4/baseline-dinov3_aux/best_model_fold7.pth
/kaggle/input/baseline-dinov3/pytorch/default/4/baseline-dinov3_aux/best_model_fold0.pth
/kaggle/input/baseline-dinov3/pytorch/default/4/baseline-dinov

In [3]:
# ============================================================================
# CELL 2: HEIGHT PREDICTION
# ============================================================================
print("\n=== STEP 1: Height Prediction ===")

height_model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None, weights_backbone=None)
path = "/kaggle/input/csiro-biomass"
local_weights = "/kaggle/input/mask-rcnn-models/pytorch/default/10/Maskrcnn_best.pt"

try:
    state_dict = torch.load(local_weights, map_location="cpu", weights_only=False)
    height_model = state_dict
    height_model.eval()
    
    for index, image_path in test_pd["image_path"].items():
        image_path = os.path.join(path, image_path)
        image = Image.open(image_path).convert("RGB")
        
        image_transform = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        image_tensor = image_transform(image)
        
        with torch.no_grad():
            outputs = height_model([image_tensor])
        
        output_pixels = outputs[0]["boxes"].cpu().numpy()
        x1, y1, x2, y2 = output_pixels[0]
        image_in_cm = y2 / 2.54
        test_pd.loc[index, "Height_Ave_cm"] = image_in_cm
    
    print("✅ Height predictions completed using Mask R-CNN")
except:
    # Fallback to mean height
    mean_height = train_pd["Height_Ave_cm"].mean()
    test_pd["Height_Ave_cm"] = mean_height
    print(f"✅ Height predictions using mean: {mean_height:.2f}")


=== STEP 1: Height Prediction ===
✅ Height predictions completed using Mask R-CNN


species model

In [4]:
import torch

torch.cuda.empty_cache()

In [5]:
# ============================================================================
# CELL 3: SPECIES PREDICTION (Optimized B0)
# ============================================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.transforms import v2
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning import Trainer
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

print("\n=== STEP 2: Species Prediction (Custom B0 + SiLU) ===")

# Global Encoders
SPECIES_LE = LabelEncoder()
TARGET_LE = LabelEncoder()

SPECIES_LE.fit(train_pd["Species"].astype(str).unique())
TARGET_LE.fit(train_pd["target_name"].astype(str).unique())

def safe_encode(le, val):
    """Encodes labels; returns 0 if label is unseen."""
    val_str = str(val)
    if val_str in le.classes_:
        return le.transform([val_str])[0]
    return 0

class SpeciesDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        tabular = torch.tensor([
            float(row["target_name"]), 
            float(row["Height_Ave_cm"])
        ], dtype=torch.float32)
        
        y = torch.tensor(int(row["Species"]), dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
        return image, tabular, y

class SpeciesDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch_size=16, num_workers=2):
        super().__init__()
        self.train_df, self.valid_df = train_df.copy(), valid_df.copy()
        self.root_dir, self.batch_size, self.num_workers = root_dir, batch_size, num_workers

    def setup(self, stage=None):
        for df in [self.train_df, self.valid_df]:
            if not np.issubdtype(df["Species"].dtype, np.number):
                df["Species"] = df["Species"].apply(lambda x: safe_encode(SPECIES_LE, x))
            if not np.issubdtype(df["target_name"].dtype, np.number):
                df["target_name"] = df["target_name"].apply(lambda x: safe_encode(TARGET_LE, x))

        self.train_tf = v2.Compose([
            v2.RandomResizedCrop(224),
            v2.RandomHorizontalFlip(),
            # REQUESTED: Added Rotation
            v2.RandomRotation(degrees=30), 
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.valid_tf = v2.Compose([
            v2.Resize(256), v2.CenterCrop(224), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        self.train_ds = SpeciesDataset(self.train_df, self.root_dir, self.train_tf)
        self.valid_ds = SpeciesDataset(self.valid_df, self.root_dir, self.valid_tf)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

class EfficientNetSpeciesClassifier(pl.LightningModule):
    def __init__(self, num_classes, target_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # Keeping B0 as requested to avoid crashing
        self.base_model = models.efficientnet_b0(weights=None)
        
        local_weights = "/kaggle/input/tf-efficientnet/pytorch/tf-efficientnet-b0/1/tf_efficientnet_b0_aa-827b6e33.pth"
        if os.path.exists(local_weights):
            state_dict = torch.load(local_weights, map_location="cpu")
            # (Weight loading logic remains same as your original snippet)
            new_state_dict = {}
            for k, v in state_dict.items():
                n = k.replace("conv_stem", "features.0.0").replace("bn1", "features.0.1")
                if "blocks" in n:
                    p = n.split(".")
                    b_idx = int(p[1]) + 1
                    sub = ".".join(p[2:])
                    if b_idx == 1:
                        sub = sub.replace("conv_dw", "block.0.0").replace("bn1", "block.0.1")
                        sub = sub.replace("se.conv_reduce", "block.1.fc1").replace("se.conv_expand", "block.1.fc2")
                        sub = sub.replace("conv_pw", "block.2.0").replace("bn2", "block.2.1")
                    else:
                        sub = sub.replace("conv_pw", "block.0.0").replace("bn1", "block.0.1")
                        sub = sub.replace("conv_dw", "block.1.0").replace("bn2", "block.1.1")
                        sub = sub.replace("se.conv_reduce", "block.2.fc1").replace("se.conv_expand", "block.2.fc2")
                        sub = sub.replace("conv_pwl", "block.3.0").replace("bn3", "block.3.1")
                    n = f"features.{b_idx}.{sub}"
                n = n.replace("conv_head", "features.8.0").replace("bn2", "features.8.1")
                new_state_dict[n] = v
            self.base_model.load_state_dict(new_state_dict, strict=False)
            print("✅ Loaded EfficientNet-B0 weights")

        self.img_dim = self.base_model.classifier[1].in_features
        self.base_model.classifier = nn.Identity()

        self.target_emb = nn.Embedding(target_dim + 1, 8)
        
        # REQUESTED: Updated to SiLU
        self.tabular_net = nn.Sequential(
            nn.Linear(8 + 1, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(), # Changed from ReLU
            nn.Dropout(0.3)
        )
        
        # REQUESTED: Bigger Head manually (added layers) + SiLU
        self.head = nn.Sequential(
            # Layer 1 (Original input)
            nn.Linear(self.img_dim + 64, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.4),
            
            # Layer 2 (New Extra Layer for "Bigger" model)
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.3),
            
            # Layer 3 (Output)
            nn.Linear(256, num_classes)
        )
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, img, target_name, height):
        img_feats = self.base_model(img)
        t_feat = self.target_emb(target_name.long())
        tab_in = torch.cat([t_feat, height.unsqueeze(1)], dim=1)
        tab_feats = self.tabular_net(tab_in)
        return self.head(torch.cat([img_feats, tab_feats], dim=1))

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.05)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=5, T_mult=1, eta_min=1e-6
        )
        return [optimizer], [scheduler]

# --- Training Section ---
image_root_dir = "/kaggle/input/csiro-biomass"
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42, stratify=train_pd["Species"])

datamodule = SpeciesDataModule(train_df=train_df, valid_df=valid_df, root_dir=image_root_dir)
datamodule.setup()

num_species = len(SPECIES_LE.classes_)
target_count = len(TARGET_LE.classes_)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

species_model = EfficientNetSpeciesClassifier(
    num_classes=num_species, 
    target_dim=target_count
).to(device)

# Safety net
early_stop_callback = EarlyStopping(
    monitor="val_loss",   
    patience=5,           
    mode="min"
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    save_top_k=1,
    mode="min"
)

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=30,        
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback],
)

trainer.fit(species_model, datamodule)

# Inference
species_model.eval()
species_model.to(device)

inf_tf = v2.Compose([
    v2.Resize(256), v2.CenterCrop(224), v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

species_results = []
with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting Species"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            img_tensor = inf_tf(img).unsqueeze(0).to(device)
            
            t_idx = safe_encode(TARGET_LE, row["target_name"])
            tab_tensor = torch.tensor([[t_idx, float(row["Height_Ave_cm"])]], dtype=torch.float32).to(device)
            
            logits = species_model(img_tensor, tab_tensor)
            class_idx = logits.argmax(dim=1).item()
            actual_name = SPECIES_LE.inverse_transform([class_idx])[0]
            species_results.append(actual_name)
        except:
            species_results.append(SPECIES_LE.classes_[0])

test_pd["Species"] = species_results
print("✅ Species predictions completed")


=== STEP 2: Species Prediction (Custom B0 + SiLU) ===
✅ Loaded EfficientNet-B0 weights


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
2026-01-19 16:59:37.297262: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768841977.462222      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768841977.509171      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768841977.918091      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768841977.918113      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin

┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ base_model  │ EfficientNet     │  4.0 M │ train │     0 │
│ 1 │ target_emb  │ Embedding        │     48 │ train │     0 │
│ 2 │ tabular_net │ Sequential       │    768 │ train │     0 │
│ 3 │ head        │ Sequential       │  825 K │ train │     0 │
│ 4 │ loss_fn     │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.8 M                                                                                                
Total estimated model params size (MB): 19                                                                         
Modules in train mode: 352                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=30` reached.


Predicting Species: 100%|██████████| 5/5 [00:00<00:00, 16.09it/s]

✅ Species predictions completed


In [6]:
import torch

torch.cuda.empty_cache()

In [7]:
# ============================================================================
# CELL 4: STATE PREDICTION
# ============================================================================
from torch.utils.data import WeightedRandomSampler
import torch.nn.functional as F
print("\n=== STEP 3: State Prediction ===")

STATE_LE = LabelEncoder()
STATE_LE.fit(train_pd["State"].astype(str).unique())

class StateDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        tabular = torch.tensor([
            float(row["target_name"]), 
            float(row["Height_Ave_cm"]), 
            float(row["Species"])
        ], dtype=torch.float32)

        y = torch.tensor(row["State"], dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
        return image, tabular, y

class StateDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch_size=16, num_workers=2):
        super().__init__()
        self.train_df, self.valid_df = train_df.copy(), valid_df.copy()
        self.root_dir, self.batch_size, self.num_workers = root_dir, batch_size, num_workers

    def setup(self, stage=None):
        for df in [self.train_df, self.valid_df]:
            if not np.issubdtype(df["State"].dtype, np.number):
                df["State"] = df["State"].apply(lambda x: safe_encode(STATE_LE, x))
            if not np.issubdtype(df["target_name"].dtype, np.number):
                df["target_name"] = df["target_name"].apply(lambda x: safe_encode(TARGET_LE, x))
            if not np.issubdtype(df["Species"].dtype, np.number):
                df["Species"] = df["Species"].apply(lambda x: safe_encode(SPECIES_LE, x))

        # --- MODIFICATION: Added RandomRotation(180) for all angles ---
        self.train_tf = v2.Compose([
            v2.RandomResizedCrop(224), 
            v2.RandomHorizontalFlip(),
            v2.RandomRotation(180),  # Rotates +/- 180 degrees
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.valid_tf = v2.Compose([
            v2.Resize(256), v2.CenterCrop(224), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.train_ds = StateDataset(self.train_df, self.root_dir, self.train_tf)
        self.valid_ds = StateDataset(self.valid_df, self.root_dir, self.valid_tf)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

class EfficientNetStateClassifier(pl.LightningModule):
    def __init__(self, num_classes, target_dim, species_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # 1. Base Model with Local Weights
        self.base_model = models.efficientnet_b0(weights=None)
        # ... [Your local weight loading logic remains here] ...

        self.img_dim = self.base_model.classifier[1].in_features
        self.base_model.classifier = nn.Identity()

        # 2. Categorical Embeddings
        self.target_emb = nn.Embedding(target_dim + 1, 8)
        self.species_emb = nn.Embedding(species_dim + 1, 12)
        
        # --- MODIFICATION: Increased Tabular Network Size & Changed to SiLU ---
        # Input is 8 (target) + 12 (species) + 1 (height) = 21
        self.tabular_net = nn.Sequential(
            nn.Linear(21, 128),      # Increased from 64
            nn.BatchNorm1d(128),
            nn.SiLU(),               # Changed ReLU to SiLU
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),      # Added extra layer
            nn.BatchNorm1d(64),
            nn.SiLU(),               # Changed ReLU to SiLU
            nn.Dropout(0.3)
        )
        
        # --- MODIFICATION: Increased Classification Head Size & Changed to SiLU ---
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 512), # Increased to 512
            nn.BatchNorm1d(512),
            nn.SiLU(),               # Changed ReLU to SiLU
            nn.Dropout(0.4),
            
            nn.Linear(512, 256),     # Added extra layer
            nn.BatchNorm1d(256),
            nn.SiLU(),               # Changed ReLU to SiLU
            nn.Dropout(0.4),
            
            nn.Linear(256, num_classes)
        )
        
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, img, target_name, height, species):
        img_feats = self.base_model(img)
        
        # Embeddings
        t_feat = self.target_emb(target_name.long())
        s_feat = self.species_emb(species.long())
        
        # Combine tabular
        tab_in = torch.cat([t_feat, height.unsqueeze(1), s_feat], dim=1)
        tab_feats = self.tabular_net(tab_in)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        # tab order: [target_name, height, species]
        logits = self(img, tab[:, 0], tab[:, 1], tab[:, 2])
        loss = self.loss_fn(logits, y)
        
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1], tab[:, 2])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.05)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        return [optimizer], [scheduler]

# Training
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42, stratify=train_pd["State"])
datamodule = StateDataModule(train_df=train_df, valid_df=valid_df, root_dir=image_root_dir)
datamodule.setup()

# 1. Calculate dimensions for the embeddings
num_states = len(STATE_LE.classes_)
target_count = len(TARGET_LE.classes_)
species_count = len(SPECIES_LE.classes_)

# 2. Initialize the model
state_model = EfficientNetStateClassifier(
    num_classes=num_states, 
    target_dim=target_count, 
    species_dim=species_count
).to(device)

# 1. Define the safety net
early_stop_callback = EarlyStopping(
    monitor="val_loss",   
    patience=5,            
    mode="min"
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    save_top_k=1,
    mode="min"
)

# 2. Configure the trainer
trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=30,        
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback],
)
trainer.fit(state_model, datamodule)

# Inference
state_model.eval()
state_model.to(device)

state_results = []
with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting State"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            img_tensor = inf_tf(img).unsqueeze(0).to(device)
            
            t_idx = safe_encode(TARGET_LE, row["target_name"])
            s_idx = safe_encode(SPECIES_LE, row["Species"])
            h_val = float(row["Height_Ave_cm"])
            
            # Create tensors for individual inputs to match forward signature
            t_tensor = torch.tensor([t_idx], dtype=torch.float32).to(device)
            s_tensor = torch.tensor([s_idx], dtype=torch.float32).to(device)
            h_tensor = torch.tensor([h_val], dtype=torch.float32).to(device)
            
            # Updated Inference Call
            logits = state_model(img_tensor, t_tensor, h_tensor, s_tensor)
            
            class_idx = logits.argmax(dim=1).item()
            actual_state = STATE_LE.inverse_transform([class_idx])[0]
            state_results.append(actual_state)
        except Exception as e:
            # print(e) # Optional debugging
            state_results.append(STATE_LE.classes_[0])

test_pd["State"] = state_results
print("✅ State predictions completed")


=== STEP 3: State Prediction ===


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ base_model  │ EfficientNet     │  4.0 M │ train │     0 │
│ 1 │ target_emb  │ Embedding        │     48 │ train │     0 │
│ 2 │ species_emb │ Embedding        │    192 │ train │     0 │
│ 3 │ tabular_net │ Sequential       │ 11.5 K │ train │     0 │
│ 4 │ head        │ Sequential       │  822 K │ train │     0 │
│ 5 │ loss_fn     │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.8 M                                                                                                
Total estimated model params size (MB): 19                                                                         
Modules in train mode: 357                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=30` reached.


Predicting State: 100%|██████████| 5/5 [00:00<00:00, 11.62it/s]

✅ State predictions completed


In [8]:
import torch

torch.cuda.empty_cache()

In [9]:
# ============================================================================
# CELL 5: NDVI PREDICTION (Modified)
# ============================================================================
print("\n=== STEP 4: NDVI Prediction ===")
import cv2
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision import models
from sklearn.model_selection import train_test_split
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning import Trainer
from tqdm import tqdm

def image_to_mask(image_path):
    """Applies HSV masking to isolate green vegetation."""
    image = cv2.imread(image_path)
    if image is None:
        return np.zeros((224, 224, 3), dtype=np.uint8)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    return cv2.bitwise_and(image, image, mask=mask)

class PreGSSHDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row["image_path"])
        masked_img = image_to_mask(img_path)
        
        s_idx = safe_encode(SPECIES_LE, row["Species"])
        t_idx = safe_encode(TARGET_LE, row["target_name"])
        ndvi_val = row["Pre_GSHH_NDVI"] if "Pre_GSHH_NDVI" in self.df.columns else 0.0
        
        tabular = torch.tensor([
            float(s_idx),
            float(ndvi_val),
            float(row["Height_Ave_cm"]),
            float(t_idx)
        ], dtype=torch.float32)

        target = torch.tensor([ndvi_val], dtype=torch.float32)

        if self.transform:
            image = self.transform(masked_img)
        else:
            image = torch.tensor(masked_img).permute(2, 0, 1).float()

        return image, tabular, target

class PreGSSHDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch=16):
        super().__init__()
        self.train_df, self.valid_df = train_df, valid_df
        self.root_dir, self.batch = root_dir, batch
        
        # ADDED: RandomRotation(180) to cover all angles
        self.tfs = v2.Compose([
            v2.ToImage(),
            v2.RandomRotation(180), # Rotates -180 to +180 degrees
            v2.Resize((224, 224)),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def setup(self, stage=None):
        self.train_ds = PreGSSHDataset(self.train_df, self.root_dir, self.tfs)
        self.valid_ds = PreGSSHDataset(self.valid_df, self.root_dir, self.tfs)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch, shuffle=True, num_workers=2)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch, num_workers=2)

class PreGSSHModel(pl.LightningModule):
    def __init__(self, species_dim, target_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # 1. Backbone with Local Weights
        self.resnet = models.resnet50(weights=None)
        local_weights = '/kaggle/input/se_resnet50/pytorch/default/1/se_resnet50-ce0d4300.pth'
        if os.path.exists(local_weights):
            state_dict = torch.load(local_weights, map_location="cpu")
            new_state_dict = {k.replace("layer0.", "").replace("last_linear", "fc"): v for k, v in state_dict.items()}
            self.resnet.load_state_dict(new_state_dict, strict=False)
            print("✅ Loaded SE-ResNet50 local weights")
        
        self.img_dim = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        # 2. Categorical Embeddings
        self.species_emb = nn.Embedding(species_dim + 1, 8)
        self.target_emb = nn.Embedding(target_dim + 1, 4)
        
        # 3. Tabular Branch (Modified: Larger + SiLU + Extra Layers)
        # Input dim is still 13 (8 emb + 4 emb + 1 height)
        self.tabular_net = nn.Sequential(
            nn.Linear(13, 128),      # Increased from 64
            nn.BatchNorm1d(128),
            nn.SiLU(),               # Changed ReLU to SiLU
            nn.Dropout(0.2),
            
            nn.Linear(128, 128),     # Added Layer
            nn.BatchNorm1d(128),
            nn.SiLU(),               # Changed ReLU to SiLU
            nn.Dropout(0.2)
        )
        
        # 4. Final Head (Modified: Larger + SiLU + Extra Layers)
        # Input dim is ResNet features + Tabular output (2048 + 128)
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 128, 512), # Increased to 512
            nn.BatchNorm1d(512),
            nn.SiLU(),               # Changed ReLU to SiLU
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),     # Added Layer
            nn.BatchNorm1d(256),
            nn.SiLU(),               # Changed ReLU to SiLU
            nn.Dropout(0.2),
            
            nn.Linear(256, 1),
            nn.Sigmoid() 
        )
        
        self.loss_fn = nn.HuberLoss(delta=0.1)

    def forward(self, img, species, target_name, height):
        img_feats = self.resnet(img)
        
        # Embeddings
        s_feat = self.species_emb(species.long())
        t_feat = self.target_emb(target_name.long())
        
        # Combine tabular (Species + Target + Height)
        tab_in = torch.cat([s_feat, t_feat, height.unsqueeze(1)], dim=1)
        tab_feats = self.tabular_net(tab_in)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        preds = self(img, tab[:, 0], tab[:, 3], tab[:, 2])
        loss = self.loss_fn(preds, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        preds = self(img, tab[:, 0], tab[:, 3], tab[:, 2])
        loss = F.mse_loss(preds, y)
        self.log("val_mse", loss, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=self.hparams.lr, 
            total_steps=self.trainer.estimated_stepping_batches
        )
        return [optimizer], [{"scheduler": scheduler, "interval": "step"}]

# Training setup
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42)
datamodule = PreGSSHDataModule(train_df, valid_df, image_root_dir)

# Inference transforms (No rotation for inference)
inf_tfs = v2.Compose([
    v2.ToImage(), 
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- Training Section ---
species_count = len(SPECIES_LE.classes_)
target_count = len(TARGET_LE.classes_)

# Initialize model
ndvi_model = PreGSSHModel(species_dim=species_count, target_dim=target_count)

early_stop_callback = EarlyStopping(
    monitor="val_mse",   
    patience=5,            
    mode="min"
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_mse",
    save_top_k=1,
    mode="min"
)

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=30,        
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback],
)
trainer.fit(ndvi_model, datamodule)

# --- Final Inference Section ---
ndvi_model.eval()
ndvi_model.to(device)

ndvi_results = []
with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting NDVI"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            masked = image_to_mask(img_path)
            img_tensor = inf_tfs(masked).unsqueeze(0).to(device)
            
            s_idx = torch.tensor([safe_encode(SPECIES_LE, row["Species"])]).to(device)
            t_idx = torch.tensor([safe_encode(TARGET_LE, row["target_name"])]).to(device)
            h_val = torch.tensor([float(row["Height_Ave_cm"])]).to(device)
            
            pred_ndvi = ndvi_model(img_tensor, s_idx, t_idx, h_val).item()
            ndvi_results.append(float(pred_ndvi))
        except Exception as e:
            ndvi_results.append(0.0)

test_pd["Pre_GSHH_NDVI"] = ndvi_results

print("✅ NDVI predictions completed")


=== STEP 4: NDVI Prediction ===


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.


✅ Loaded SE-ResNet50 local weights


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ resnet      │ ResNet     │ 23.5 M │ train │     0 │
│ 1 │ species_emb │ Embedding  │    128 │ train │     0 │
│ 2 │ target_emb  │ Embedding  │     24 │ train │     0 │
│ 3 │ tabular_net │ Sequential │ 18.8 K │ train │     0 │
│ 4 │ head        │ Sequential │  1.2 M │ train │     0 │
│ 5 │ loss_fn     │ HuberLoss  │      0 │ train │     0 │
└───┴─────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 24.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.8 M                                                                                               
Total estimated model params size (MB): 99                                                                         
Modules in train mode: 174                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=30` reached.


Predicting NDVI: 100%|██████████| 5/5 [00:00<00:00,  9.47it/s]

✅ NDVI predictions completed


In [10]:
import torch

torch.cuda.empty_cache()

In [11]:
# ============================================================================
# CELL 6: FINAL BIOMASS PREDICTION (UPDATED - SCALED R2 LOSS)
# ============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning import Trainer
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from PIL import Image
import os
import numpy as np
from tqdm.auto import tqdm

print("\n=== STEP 5: Final Biomass Prediction (Scaled R2 + Enhanced) ===")

# --- 1. Custom SCALED R2 Loss Function ---
class ScaledR2Loss(nn.Module):
    """
    Calculates R2 Loss on scaled data (Z-scores) to stabilize training 
    when targets have large ranges (e.g., 0-3000).
    
    Logic:
    1. Normalize preds and targets using the batch's mean/std.
    2. Calculate MSE on these normalized values.
    3. This effectively optimizes for R2 without exploding gradients from large values.
    """
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, preds, target):
        # Calculate statistics for the current batch
        target_mean = target.mean()
        target_std = target.std() + self.eps # Avoid div by zero
        
        # Normalize (Scale) Target and Prediction to Z-scores
        # We want the model to predict the *relative* position in the distribution
        t_norm = (target - target_mean) / target_std
        p_norm = (preds - target_mean) / target_std 
        
        # Now calculate R2-style loss on normalized values
        # Loss = SS_res_norm / SS_tot_norm
        ss_res_norm = torch.sum((t_norm - p_norm) ** 2)
        ss_tot_norm = torch.sum((t_norm - t_norm.mean()) ** 2)
        
        # If the batch has 0 variance (all same values), fallback to standard MSE
        if ss_tot_norm < self.eps:
            return F.mse_loss(preds, target)
            
        return ss_res_norm / (ss_tot_norm + self.eps)

# --- 2. Encoders & Dataset ---
TARGET_NAME_LE = LabelEncoder()
TARGET_NAME_LE.fit(train_pd["target_name"].astype(str).unique())

class BiomassDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            img = Image.open(image_path).convert("RGB")
        except:
            img = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform:
            img = self.transform(img)

        tab = torch.tensor([
            float(safe_encode(STATE_LE, row["State"])),
            float(safe_encode(SPECIES_LE, row["Species"])),
            float(row["Pre_GSHH_NDVI"]),
            float(row["Height_Ave_cm"]),
            float(safe_encode(TARGET_NAME_LE, row["target_name"]))
        ], dtype=torch.float32)

        if self.is_train:
            target = torch.tensor(row["target"], dtype=torch.float32)
            return img, tab, target
        else:
            return img, tab

# --- 3. Enhanced Model Architecture (SiLU + Larger) ---
class BiomassLightningModel(pl.LightningModule):
    def __init__(self, state_dim, species_dim, target_name_dim, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        # A. Backbone
        self.resnet = models.resnet50(weights=None)
        local_weights = '/kaggle/input/se_resnet50/pytorch/default/1/se_resnet50-ce0d4300.pth'
        
        if os.path.exists(local_weights):
            state_dict = torch.load(local_weights, map_location="cpu")
            new_state_dict = {k.replace("layer0.", "").replace("last_linear", "fc"): v for k, v in state_dict.items()}
            self.resnet.load_state_dict(new_state_dict, strict=False)
            print("✅ SE-ResNet50 Weights Loaded")

        self.img_dim = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        # B. Embeddings
        self.state_emb = nn.Embedding(state_dim + 1, 8)
        self.species_emb = nn.Embedding(species_dim + 1, 16)
        self.target_name_emb = nn.Embedding(target_name_dim + 1, 8)
        
        # C. Larger Tabular Network
        self.tab_net = nn.Sequential(
            nn.Linear(34, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),             # SiLU Activation
            nn.Dropout(0.3),
            
            nn.Linear(128, 256),   # Extra Layer
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 64)
        )
        
        # D. Larger Fusion Head
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 1024),
            nn.BatchNorm1d(1024),
            nn.SiLU(),             # SiLU Activation
            nn.Dropout(0.4),
            
            nn.Linear(1024, 512),  # Extra Layer
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),   # Extra Layer
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 1)
        )

        # Loss Function
        self.loss_fn = ScaledR2Loss()

        # Initialization
        for m in [self.state_emb, self.species_emb, self.target_name_emb, self.tab_net, self.head]:
            for layer in m.modules():
                if isinstance(layer, nn.Linear):
                    nn.init.kaiming_normal_(layer.weight, mode='fan_out', nonlinearity='relu')
                elif isinstance(layer, nn.BatchNorm1d):
                    nn.init.constant_(layer.weight, 1)
                    nn.init.constant_(layer.bias, 0)

    def forward(self, img, state, species, target_name, ndvi_height):
        img_feats = self.resnet(img)
        
        s_emb = self.state_emb(state.long())
        sp_emb = self.species_emb(species.long())
        t_emb = self.target_name_emb(target_name.long())
        
        tab_combined = torch.cat([s_emb, sp_emb, t_emb, ndvi_height], dim=1)
        tab_feats = self.tab_net(tab_combined)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined).squeeze(1)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        state, species, ndvi, height, t_name = tab[:,0], tab[:,1], tab[:,2], tab[:,3], tab[:,4]
        ndvi_height = tab[:, 2:4]
        
        preds = self(img, state, species, t_name, ndvi_height)
        
        # Use SCALED R2 Loss
        loss = self.loss_fn(preds, y)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        state, species, ndvi, height, t_name = tab[:,0], tab[:,1], tab[:,2], tab[:,3], tab[:,4]
        ndvi_height = tab[:, 2:4]
        
        preds = self(img, state, species, t_name, ndvi_height)
        
        loss = F.mse_loss(preds, y)
        self.log("val_mse", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.05)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=self.hparams.lr, 
            total_steps=self.trainer.estimated_stepping_batches
        )
        return [optimizer], [{"scheduler": scheduler, "interval": "step"}]

# --- 4. Transforms (Rotations Added) ---
train_df, valid_df = train_test_split(train_pd, test_size=0.15, random_state=42)

biomass_tfs = v2.Compose([
    v2.Resize((224, 224)),
    # Rotate all angles (0-180)
    v2.RandomRotation(degrees=180), 
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = BiomassDataset(train_df, image_root_dir, biomass_tfs, is_train=True)
valid_ds = BiomassDataset(valid_df, image_root_dir, biomass_tfs, is_train=True)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=16, num_workers=2)

# --- 5. Training ---
num_states = len(STATE_LE.classes_)
num_species = len(SPECIES_LE.classes_)
num_targets = len(TARGET_NAME_LE.classes_)

biomass_model = BiomassLightningModel(
    state_dim=num_states, 
    species_dim=num_species, 
    target_name_dim=num_targets
)

# Callbacks
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

early_stop_callback = EarlyStopping(monitor="val_mse", patience=7, mode="min", verbose=True)
checkpoint_callback = ModelCheckpoint(
    dirpath="/kaggle/working/checkpoints",
    filename="best-biomass-model",
    monitor="val_mse",
    save_top_k=1,
    mode="min"
)

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=40, 
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback]
)

trainer.fit(biomass_model, train_loader, valid_loader)

# --- 6. Inference ---
biomass_model.eval()
biomass_model.to(device)
final_results = []

# Inference Transform (Clean)
clean_tfs = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            
            img_tensor = clean_tfs(img).unsqueeze(0).to(device)
            
            s_val = torch.tensor([safe_encode(STATE_LE, row["State"])]).to(device)
            sp_val = torch.tensor([safe_encode(SPECIES_LE, row["Species"])]).to(device)
            t_val = torch.tensor([safe_encode(TARGET_NAME_LE, row["target_name"])]).to(device)
            nh_val = torch.tensor([[float(row["Pre_GSHH_NDVI"]), float(row["Height_Ave_cm"])]], dtype=torch.float32).to(device)
            
            pred_biomass = biomass_model(img_tensor, s_val, sp_val, t_val, nh_val).item()
            final_results.append(max(0.0, pred_biomass))
        except Exception as e:
            final_results.append(0.0)

submission_df = pd.DataFrame({
    "sample_id": test_pd_original["sample_id"],
    "target": final_results
})
submission_df.to_csv('/kaggle/working/submission.csv', index=False)
print(submission_df.head())


=== STEP 5: Final Biomass Prediction (Scaled R2 + Enhanced) ===


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.


✅ SE-ResNet50 Weights Loaded


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ resnet          │ ResNet       │ 23.5 M │ train │     0 │
│ 1 │ state_emb       │ Embedding    │     40 │ train │     0 │
│ 2 │ species_emb     │ Embedding    │    256 │ train │     0 │
│ 3 │ target_name_emb │ Embedding    │     48 │ train │     0 │
│ 4 │ tab_net         │ Sequential   │ 54.7 K │ train │     0 │
│ 5 │ head            │ Sequential   │  2.8 M │ train │     0 │
│ 6 │ loss_fn         │ ScaledR2Loss │      0 │ train │     0 │
└───┴─────────────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 26.4 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 26.4 M                                                                                               
Total estimated model params size (MB): 105                                                                        
Modules in train mode: 179                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val_mse improved. New best score: 1210.087
Metric val_mse improved by 346.269 >= min_delta = 0.0. New best score: 863.818
Metric val_mse improved by 428.321 >= min_delta = 0.0. New best score: 435.497
Metric val_mse improved by 38.712 >= min_delta = 0.0. New best score: 396.785
Metric val_mse improved by 135.702 >= min_delta = 0.0. New best score: 261.083
Metric val_mse improved by 56.285 >= min_delta = 0.0. New best score: 204.798
Metric val_mse improved by 26.075 >= min_delta = 0.0. New best score: 178.723
Metric val_mse improved by 11.995 >= min_delta = 0.0. New best score: 166.729
Metric val_mse improved by 26.185 >= min_delta = 0.0. New best score: 140.544
Monitored metric val_mse did not improve in the last 7 records. Best score: 140.544. Signaling Trainer to stop.


Predicting:   0%|          | 0/5 [00:00<?, ?it/s]

                    sample_id      target
0  ID1001187975__Dry_Clover_g  132.586227
1    ID1001187975__Dry_Dead_g  117.867767
2   ID1001187975__Dry_Green_g  179.283554
3   ID1001187975__Dry_Total_g  194.772888
4         ID1001187975__GDM_g  190.195663
